# Agent SQL sur le catalogue : recherche sémantique, lineage, exécution lecture seule

Notebook **autonome** : aucune dépendance à `src/`, `docmaker/` ou à
`semantic_layer.ipynb` — tout, y compris les données de démonstration, vit ici.
Objectif : comprendre comment câbler un LLM à des outils (*tool calling*) pour
répondre à des questions sur un datamart, et comparer trois façons de l'orchestrer.

## Le modèle : OpenRouter ou Ollama, indifféremment

Les deux exposent une API **compatible OpenAI** (`/v1/chat/completions`, y compris
le tool calling) : un seul client (`openai.OpenAI`), seul `base_url` change. Basculer
de l'un à l'autre est une ligne dans la cellule de configuration — aucune autre
cellule ne le sait. Pour Ollama, choisir un modèle qui supporte le tool calling
(`qwen2.5`, `llama3.1`, `mistral-nemo`... — pas tous les modèles le font).

## Les outils de l'agent

| Outil                | Rôle                                                                       |
| --------------------- | ---------------------------------------------------------------------------- |
| `search_catalog`      | recherche sémantique table/colonne (version compacte de `semantic_layer.ipynb`) |
| `get_table_metadata`  | colonnes, types, clés, commentaires d'une table                             |
| `get_lineage`         | expression de calcul d'une colonne dérivée + arêtes de jointure             |
| `run_sql`             | exécution **lecture seule**, plafonnée en lignes et en temps                |

`run_sql` sert aux deux usages demandés : l'agent l'utilise pour explorer
(`SELECT * FROM x LIMIT 5`, compter des valeurs distinctes...) et pour produire la
réponse finale (l'agrégat qui répond réellement à la question). C'est le même outil,
le même garde-fou ; seul l'usage diffère.

## Trois approches, comparées sur les mêmes outils

| Approche | Framework  | Liberté du modèle                                                          |
| -------- | ---------- | ------------------------------------------------------------------------- |
| **A**    | aucun      | totale : boucle *tool calling* manuelle (ReAct maison)                    |
| **B**    | LangGraph  | totale : le même agent, porté nœud pour nœud sur `StateGraph`             |
| **C**    | LangGraph  | encadrée : pipeline déterministe (retrouve → génère → valide → exécute), le modèle ne remplit que des cases précises — inspiré de `docmaker/eval/runtime.py` du dépôt |

La section 8 compare les trois. **B ne fait rien de plus que A** — c'est volontaire,
pour isoler ce que LangGraph apporte réellement, qui se voit en C : un contrôle de
flux explicite (retry borné, abstention motivée) au lieu d'un modèle livré à lui-même.

## Données de démonstration

Un mini-datamart bancaire en mémoire (SQLite), cohérent avec `semantic_layer.ipynb` :
comptes, mouvements, soldes, encours de crédit, taux de change, adresses. Bascule
possible vers un vrai Oracle en lecture seule (`DB_BACKEND = "oracle"`, section 3).

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sqlite3
import threading
from pathlib import Path

import numpy as np
import sqlglot
from dotenv import load_dotenv
from openai import OpenAI
from sqlglot import exp

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
log = logging.getLogger("sql_agent")

# ============================================================================
# CONFIGURATION -- tout ce qui est tweakable est ici.
# ============================================================================

PROVIDER = "ollama"  # "openrouter" (cloud) | "ollama" (local) -- meme code, meme API
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-haiku-4.5")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")  # doit supporter le tool calling
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

DB_BACKEND = "demo_sqlite"  # "demo_sqlite" (aucune infra) | "oracle" (datamart reel)
ORACLE_DSN = os.environ.get("ORACLE_DSN", "hote:1521/service")
ORACLE_USER = os.environ.get("ORACLE_USER", "compte_lecture")

MAX_RESULT_ROWS = 50          # plafond dur applique a toute execution SQL
QUERY_TIMEOUT_SECONDS = 10    # meilleur effort en sqlite (section 3), reel sous Oracle
MAX_TOOL_ROUNDS = 8           # borne dure sur la boucle agent (anti boucle infinie), approches A/B
MAX_SQL_ATTEMPTS = 3          # borne sur les tentatives de generation SQL, approche C
SEMANTIC_TOP_K = 5


def get_client_and_model() -> tuple[OpenAI, str]:
    if PROVIDER == "openrouter":
        return (
            OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["LLM_API_KEY"]),
            OPENROUTER_MODEL,
        )
    if PROVIDER == "ollama":
        # cle API factice : le SDK openai l'exige, Ollama l'ignore.
        return OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama"), OLLAMA_MODEL
    raise ValueError(f"PROVIDER inconnu : {PROVIDER!r}")


client, MODEL = get_client_and_model()

## 1. Données de démonstration : le mini-datamart

SQLite en mémoire : assez de lignes pour que `run_sql` renvoie de vrais résultats,
pas assez pour masquer les erreurs de l'agent derrière du bruit. Mêmes tables que
`semantic_layer.ipynb` (noms en minuscules, convention SQLite).

In [ ]:
def build_demo_database() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.executescript("""
        CREATE TABLE dmt_cpt_mvt_j (
            id_mvt INTEGER PRIMARY KEY, id_compte INTEGER, dt_mvt TEXT,
            cd_typ_ope TEXT, mt_mvt REAL
        );
        CREATE TABLE dmt_cpt_sld_j (
            id_compte INTEGER, dt_jour TEXT, mt_sld_chf REAL
        );
        CREATE TABLE ods_d_cli_adr (
            id_client INTEGER, rue TEXT, ville TEXT, dt_deb_val TEXT
        );
        CREATE TABLE dmt_f_crd_enc_m (
            id_dossier INTEGER, id_client INTEGER, dt_fin_mois TEXT, mt_crd_restant REAL
        );
        CREATE TABLE v_ref_fin_txc_chf (
            devise TEXT, dt_jour TEXT, tx_chf REAL
        );
    """)
    rows = {
        "dmt_cpt_mvt_j": [
            (1, 101, "2026-09-01", "debit", -120.50), (2, 101, "2026-09-02", "credit", 500.00),
            (3, 102, "2026-09-01", "debit", -40.00), (4, 102, "2026-09-03", "debit", -15.90),
            (5, 103, "2026-09-02", "credit", 1200.00), (6, 101, "2026-09-05", "debit", -60.00),
            (7, 102, "2026-09-05", "virement", -200.00), (8, 103, "2026-09-06", "debit", -75.30),
        ],
        "dmt_cpt_sld_j": [
            (101, "2026-09-05", 2340.75), (102, "2026-09-05", 890.10), (103, "2026-09-05", 15420.00),
        ],
        "ods_d_cli_adr": [
            (201, "Rue du Lac 4", "Geneve", "2020-01-01"),
            (202, "Bahnhofstrasse 12", "Zurich", "2019-06-15"),
            (203, "Via Nassa 9", "Lugano", "2021-03-10"),
        ],
        "dmt_f_crd_enc_m": [
            (301, 201, "2026-08-31", 45000.00), (302, 202, "2026-08-31", 128900.00),
            (303, 203, "2026-08-31", 0.0),
        ],
        "v_ref_fin_txc_chf": [
            ("EUR", "2026-09-05", 0.96), ("USD", "2026-09-05", 0.88), ("CHF", "2026-09-05", 1.0),
        ],
    }
    for table, values in rows.items():
        placeholders = ", ".join("?" * len(values[0]))
        conn.executemany(f"INSERT INTO {table} VALUES ({placeholders})", values)
    conn.commit()
    return conn


DEMO_DB = build_demo_database() if DB_BACKEND == "demo_sqlite" else None

## 2. Métadonnées et lineage : de simples dictionnaires

En production, ces deux structures sont des artefacts produits par le pipeline
(`docmaker/pipeline/catalog.py` -> `build/catalog.json`,
`docmaker/pipeline/lineage.py` -> `build/lineage.json`, en parsant les vues Oracle
avec `sqlglot`). Ici, écrits à la main pour rester autonome : mêmes clés que ces
artefacts réels, pour qu'un vrai `catalog.json`/`lineage.json` se substitue à
`CATALOG`/`LINEAGE_COLUMNS` sans changer les outils qui les consomment.

In [ ]:
CATALOG: dict[str, dict] = {
    "DMT.DMT_CPT_MVT_J": {
        "sqlite_table": "dmt_cpt_mvt_j",
        "comment": "Mouvements comptables journaliers par compte.",
        "columns": [
            {"name": "id_mvt", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "id_compte", "type": "NUMBER", "key": "FK",
             "comment": "-> comptes (implicite, non declare)"},
            {"name": "dt_mvt", "type": "DATE", "key": "", "comment": ""},
            {"name": "cd_typ_ope", "type": "VARCHAR2", "key": "", "comment": "debit | credit | virement"},
            {"name": "mt_mvt", "type": "NUMBER", "key": "", "comment": "Montant signe, devise d'origine."},
        ],
    },
    "DMT.DMT_CPT_SLD_J": {
        "sqlite_table": "dmt_cpt_sld_j",
        "comment": "Solde de fin de journee par compte, en CHF.",
        "columns": [
            {"name": "id_compte", "type": "NUMBER", "key": "FK", "comment": ""},
            {"name": "dt_jour", "type": "DATE", "key": "", "comment": ""},
            {"name": "mt_sld_chf", "type": "NUMBER", "key": "", "comment": "Cumul des mouvements, voir lineage."},
        ],
    },
    "ODS.ODS_D_CLI_ADR": {
        "sqlite_table": "ods_d_cli_adr",
        "comment": "Adresses postales des clients, historisees.",
        "columns": [
            {"name": "id_client", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "rue", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "ville", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "dt_deb_val", "type": "DATE", "key": "", "comment": "Debut de validite de l'adresse."},
        ],
    },
    "DMT.DMT_F_CRD_ENC_M": {
        "sqlite_table": "dmt_f_crd_enc_m",
        "comment": "Encours de credit mensuels par dossier.",
        "columns": [
            {"name": "id_dossier", "type": "NUMBER", "key": "PK", "comment": ""},
            {"name": "id_client", "type": "NUMBER", "key": "FK", "comment": "-> ods_d_cli_adr.id_client"},
            {"name": "dt_fin_mois", "type": "DATE", "key": "", "comment": ""},
            {"name": "mt_crd_restant", "type": "NUMBER", "key": "", "comment": "Capital restant du en fin de mois."},
        ],
    },
    "REF.V_REF_FIN_TXC_CHF": {
        "sqlite_table": "v_ref_fin_txc_chf",
        "comment": "Cours de conversion quotidiens vers le franc suisse.",
        "columns": [
            {"name": "devise", "type": "VARCHAR2", "key": "", "comment": ""},
            {"name": "dt_jour", "type": "DATE", "key": "", "comment": ""},
            {"name": "tx_chf", "type": "NUMBER", "key": "", "comment": ""},
        ],
    },
}

# Lineage colonne a colonne : ce qu'un vrai lineage.py extrairait en parsant les vues
# Oracle. Ici a la main, sur la seule colonne derivee du mini-datamart : le solde
# n'est pas stocke tel quel, c'est un cumul des mouvements.
LINEAGE_COLUMNS = [
    {
        "target": "DMT.DMT_CPT_SLD_J.MT_SLD_CHF",
        "expression": "SUM(m.mt_mvt) OVER (PARTITION BY m.id_compte ORDER BY m.dt_mvt)",
        "sources": ["DMT.DMT_CPT_MVT_J.MT_MVT", "DMT.DMT_CPT_MVT_J.ID_COMPTE"],
        "object_fqn": "DMT.DMT_CPT_SLD_J",
    },
]

# Aretes de jointure observees (ou declarees) entre tables.
JOIN_EDGES = [
    {"left": "DMT.DMT_CPT_MVT_J.ID_COMPTE", "right": "DMT.DMT_CPT_SLD_J.ID_COMPTE", "frequency": 1},
    {"left": "DMT.DMT_F_CRD_ENC_M.ID_CLIENT", "right": "ODS.ODS_D_CLI_ADR.ID_CLIENT", "frequency": 1},
]

_SQLITE_TO_FQN = {meta["sqlite_table"]: fqn for fqn, meta in CATALOG.items()}

## 3. Les outils

### `search_catalog` — recherche sémantique

Version compacte à **un seul canal** (le texte complet de chaque table/colonne
embeddé tel quel), pour rester courte : voir `semantic_layer.ipynb` pour la version
à deux canaux identité/sens, plus fine mais plus longue à mettre en place. Le
principe (cosinus via produit scalaire numpy) est identique.

In [ ]:
from fastembed import TextEmbedding

_EMBEDDER = TextEmbedding("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def _catalog_documents() -> list[dict]:
    docs = []
    for fqn, meta in CATALOG.items():
        cols = ", ".join(c["name"] for c in meta["columns"])
        docs.append({
            "fqn": fqn, "entity_type": "table",
            "text": f"table {fqn} ({meta['comment']}) colonnes: {cols}",
        })
        for c in meta["columns"]:
            docs.append({
                "fqn": f"{fqn}.{c['name']}", "entity_type": "column",
                "text": f"colonne {c['name']} de {fqn} : {c['comment'] or c['type']}",
            })
    return docs


_CATALOG_DOCS = _catalog_documents()
_CATALOG_VECTORS = np.array(list(_EMBEDDER.embed([d["text"] for d in _CATALOG_DOCS])))
_CATALOG_VECTORS = _CATALOG_VECTORS / np.linalg.norm(_CATALOG_VECTORS, axis=1, keepdims=True)


def search_catalog(query: str, top_k: int = SEMANTIC_TOP_K, entity_type: str | None = None) -> list[dict]:
    """Recherche semantique table/colonne par cosinus (produit scalaire sur vecteurs
    normes). Renvoie les FQN les plus proches, avec leur score.
    """
    vector = next(iter(_EMBEDDER.embed([query])))
    vector = vector / np.linalg.norm(vector)
    scores = _CATALOG_VECTORS @ vector
    order = np.argsort(-scores)
    hits = []
    for i in order:
        doc = _CATALOG_DOCS[i]
        if entity_type and doc["entity_type"] != entity_type:
            continue
        hits.append({"fqn": doc["fqn"], "entity_type": doc["entity_type"], "score": round(float(scores[i]), 3)})
        if len(hits) >= top_k:
            break
    return hits

### `get_table_metadata` — l'équivalent de `build/catalog.json` pour une table

In [ ]:
def get_table_metadata(fqn: str) -> dict:
    """Colonnes, types, cles, commentaire et nombre de lignes de la table demandee."""
    meta = CATALOG.get(fqn.upper())
    if meta is None:
        return {"error": f"table inconnue : {fqn!r}. Tables disponibles : {list(CATALOG)}"}
    row_count = None
    if DB_BACKEND == "demo_sqlite":
        row_count = DEMO_DB.execute(f"SELECT COUNT(*) FROM {meta['sqlite_table']}").fetchone()[0]
    return {"fqn": fqn.upper(), "comment": meta["comment"], "row_count": row_count, "columns": meta["columns"]}

### `get_lineage` — d'où vient une colonne, avec quoi se joint-elle

In [ ]:
def get_lineage(table_or_column: str) -> dict:
    """Cherche `table_or_column` (sous-chaine, insensible a la casse) parmi les
    cibles/sources du lineage colonne a colonne, et parmi les aretes de jointure.
    """
    needle = table_or_column.upper()
    columns = [
        c for c in LINEAGE_COLUMNS
        if needle in c["target"] or any(needle in s for s in c["sources"])
    ]
    joins = [j for j in JOIN_EDGES if needle in j["left"] or needle in j["right"]]
    if not columns and not joins:
        return {"message": f"aucun lineage trouve pour {table_or_column!r}"}
    return {"column_lineage": columns, "joins": joins}

### `run_sql` — exécution lecture seule

Le seul garde-fou non négociable (même esprit que `config.toml [validator]` et
`docmaker/semantic/validate.py` du dépôt) : **SELECT uniquement**, une seule
instruction, résultat plafonné en lignes. Le reste (forme exacte du SQL) est laissé
libre — l'agent peut se tromper et corriger, c'est le but.

In [ ]:
class ReadOnlyViolation(Exception):
    pass


def _ensure_select_only(sql: str, dialect: str) -> None:
    try:
        statements = sqlglot.parse(sql, dialect=dialect)
    except Exception as e:
        raise ReadOnlyViolation(f"SQL non parsable : {e}") from e
    if len(statements) != 1 or statements[0] is None:
        raise ReadOnlyViolation("une seule instruction a la fois")
    if not isinstance(statements[0], exp.Select):
        raise ReadOnlyViolation("seules les requetes SELECT sont autorisees")


def _cap_rows_sqlite(sql: str, limit: int) -> str:
    return f"SELECT * FROM ({sql}) AS capped LIMIT {limit}"


def _run_sqlite(sql: str, limit: int, timeout_s: float) -> dict:
    """`threading` fournit un budget de temps *best-effort* : sqlite ne peut pas etre
    annule proprement depuis un autre thread, la requete continue en arriere-plan si
    elle depasse le budget (limite assumee de ce backend de demo -- Oracle, lui,
    annule reellement via `call_timeout`, voir `_run_oracle`).
    """
    result: dict = {}

    def worker():
        try:
            cur = DEMO_DB.execute(_cap_rows_sqlite(sql, limit))
            cols = [d[0] for d in cur.description]
            result["columns"] = cols
            result["rows"] = [dict(zip(cols, r, strict=True)) for r in cur.fetchall()]
        except Exception as e:  # noqa: BLE001 -- renvoye a l'agent, pas leve
            result["error"] = str(e)

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(timeout_s)
    if thread.is_alive():
        return {"error": f"depassement du budget de {timeout_s}s (backend demo, non annulable)"}
    return result


def run_sql(sql: str) -> dict:
    """Execute `sql` en lecture seule, plafonne en lignes (`MAX_RESULT_ROWS`) et en
    temps (`QUERY_TIMEOUT_SECONDS`). Sert aussi bien a explorer (l'agent peut essayer,
    se tromper, corriger) qu'a produire la reponse finale : meme outil, meme garde-fou.
    """
    dialect = "oracle" if DB_BACKEND == "oracle" else "sqlite"
    try:
        _ensure_select_only(sql, dialect)
    except ReadOnlyViolation as e:
        return {"error": str(e)}

    if DB_BACKEND == "demo_sqlite":
        outcome = _run_sqlite(sql, MAX_RESULT_ROWS, QUERY_TIMEOUT_SECONDS)
    elif DB_BACKEND == "oracle":
        outcome = _run_oracle(sql, MAX_RESULT_ROWS, QUERY_TIMEOUT_SECONDS)
    else:
        raise ValueError(f"DB_BACKEND inconnu : {DB_BACKEND!r}")

    if "error" in outcome:
        return outcome
    truncated = len(outcome["rows"]) >= MAX_RESULT_ROWS
    return {"columns": outcome["columns"], "rows": outcome["rows"], "truncated": truncated}

### Bascule vers un vrai Oracle en lecture seule

Reprend `docmaker/pipeline/_oracle.py::Db` : import différé (`oracledb` n'est
nécessaire que sur ce chemin), timeout serveur réel via `call_timeout`, mot de passe
lu depuis `.env` (jamais en dur). N'est appelée que si `DB_BACKEND = "oracle"`.

In [ ]:
def _run_oracle(sql: str, limit: int, timeout_s: float) -> dict:
    import oracledb

    capped = f"SELECT * FROM ({sql}) FETCH FIRST {limit} ROWS ONLY"
    password = os.environ.get("ORACLE_PASSWORD")
    if not password:
        return {"error": "ORACLE_PASSWORD absent de l'environnement (.env)"}
    try:
        with oracledb.connect(user=ORACLE_USER, password=password, dsn=ORACLE_DSN) as conn:
            conn.call_timeout = int(timeout_s * 1000)
            with conn.cursor() as cur:
                cur.execute(capped)
                cols = [d[0].lower() for d in cur.description]
                rows = [dict(zip(cols, r, strict=True)) for r in cur.fetchall()]
                return {"columns": cols, "rows": rows}
    except Exception as e:  # noqa: BLE001 -- renvoye a l'agent
        return {"error": str(e)}

### Déclarer les outils au format OpenAI (`tools=[...]`) et les brancher

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_catalog",
        "description": "Recherche semantique de tables/colonnes pertinentes pour une question en langage naturel.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "default": SEMANTIC_TOP_K},
                "entity_type": {"type": "string", "enum": ["table", "column"]},
            },
            "required": ["query"],
        },
    }},
    {"type": "function", "function": {
        "name": "get_table_metadata",
        "description": "Colonnes, types, cles et commentaire d'une table (FQN 'SCHEMA.TABLE').",
        "parameters": {
            "type": "object",
            "properties": {"fqn": {"type": "string"}},
            "required": ["fqn"],
        },
    }},
    {"type": "function", "function": {
        "name": "get_lineage",
        "description": "Expression de calcul d'une colonne derivee et jointures connues pour une table ou colonne.",
        "parameters": {
            "type": "object",
            "properties": {"table_or_column": {"type": "string"}},
            "required": ["table_or_column"],
        },
    }},
    {"type": "function", "function": {
        "name": "run_sql",
        "description": (
            "Execute une requete SQL en LECTURE SEULE (SELECT uniquement, plafonnee "
            f"a {MAX_RESULT_ROWS} lignes). A utiliser aussi bien pour explorer les "
            "donnees que pour calculer la reponse finale a la question."
        ),
        "parameters": {
            "type": "object",
            "properties": {"sql": {"type": "string"}},
            "required": ["sql"],
        },
    }},
]

TOOL_FUNCS = {
    "search_catalog": search_catalog,
    "get_table_metadata": get_table_metadata,
    "get_lineage": get_lineage,
    "run_sql": run_sql,
}


def dispatch_tool_call(name: str, arguments: dict) -> str:
    """Execute un appel d'outil et renvoie du JSON -- jamais d'exception : une erreur
    d'outil est un resultat que l'agent doit voir et sur lequel reagir, pas un crash.
    """
    func = TOOL_FUNCS.get(name)
    if func is None:
        return json.dumps({"error": f"outil inconnu : {name!r}"})
    try:
        return json.dumps(func(**arguments), default=str, ensure_ascii=False)
    except Exception as e:  # noqa: BLE001
        return json.dumps({"error": str(e)}, ensure_ascii=False)

In [ ]:
# Sanity check direct (sans LLM) : chaque outil repond bien quelque chose de sense.
print(search_catalog("solde journalier d'un compte"))
print(get_table_metadata("DMT.DMT_CPT_SLD_J"))
print(get_lineage("MT_SLD_CHF"))
print(run_sql("SELECT id_compte, SUM(mt_mvt) AS total FROM dmt_cpt_mvt_j GROUP BY id_compte"))

## 4. Approche A — boucle agent manuelle (sans framework)

Le cœur d'un agent *tool calling*, sans aucune bibliothèque : une liste de messages,
un appel modèle, et une branche — soit le modèle répond du texte (fini), soit il
demande des outils (on les exécute, on ajoute les résultats aux messages, on
rappelle le modèle). `MAX_TOOL_ROUNDS` borne la boucle si le modèle n'en finit pas.

In [ ]:
SYSTEM_PROMPT = (
    "Tu es un assistant analytique sur un datamart bancaire. Utilise les outils a "
    "disposition pour explorer le catalogue, comprendre le lineage et interroger les "
    "donnees. Ne reponds jamais une valeur chiffree sans l'avoir verifiee par `run_sql`. "
    "Si les outils ne permettent pas de repondre, dis-le explicitement."
)


def run_agent_manual(question: str, verbose: bool = True) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

    for round_ in range(MAX_TOOL_ROUNDS):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content or ""

        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments or "{}")
            if verbose:
                print(f"[round {round_}] {call.function.name}({arguments})")
            result = dispatch_tool_call(call.function.name, arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "abstention : nombre maximal d'allers-retours outils atteint"

In [ ]:
print(run_agent_manual("Quel est le solde actuel du compte 101 ?"))

In [ ]:
print(run_agent_manual("D'ou vient la colonne mt_sld_chf de DMT_CPT_SLD_J, et avec quoi se joint-elle ?"))

## 5. Approche B — le même agent avec LangGraph

Nécessite `langgraph` (`poetry add langgraph`, ou `pip install langgraph` hors
poetry). Le même comportement que l'approche A, exprimé comme un graphe : un nœud
`agent` (appelle le modèle), un nœud `tools` (exécute les appels demandés), une
arête conditionnelle qui boucle tant que le modèle demande des outils.

**Choix volontaire** : l'état ne stocke que des dicts au format OpenAI brut (comme en
A), fusionnés par un reducer `operator.add` sur une liste — pas le reducer
`add_messages` de LangGraph, pensé pour les types de messages LangChain
(`HumanMessage`, `AIMessage`...). Aucun wrapper LangChain (`ChatOpenAI`) non plus :
`call_model` appelle le même client `openai` que l'approche A. Ça garde la comparaison
A/B à isocode, et ça montre que LangGraph s'utilise avec n'importe quelle fonction
Python comme nœud — pas besoin de ses intégrations LLM pour en profiter.

In [ ]:
try:
    import operator
    from typing import Annotated, TypedDict

    from langgraph.graph import END, START, StateGraph
except ImportError as e:
    raise ImportError(
        "LangGraph n'est pas installe dans cet environnement. "
        "Poetry : `poetry add langgraph`. Sinon : `pip install langgraph`."
    ) from e

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[dict], operator.add]


def call_model(state: AgentState) -> dict:
    response = client.chat.completions.create(model=MODEL, messages=state["messages"], tools=TOOLS)
    return {"messages": [response.choices[0].message.model_dump(exclude_none=True)]}


def call_tools(state: AgentState) -> dict:
    last = state["messages"][-1]
    results = []
    for call in last["tool_calls"]:
        arguments = json.loads(call["function"]["arguments"] or "{}")
        content = dispatch_tool_call(call["function"]["name"], arguments)
        results.append({"role": "tool", "tool_call_id": call["id"], "content": content})
    return {"messages": results}


def should_continue(state: AgentState) -> str:
    return "tools" if state["messages"][-1].get("tool_calls") else END


agent_graph_builder = StateGraph(AgentState)
agent_graph_builder.add_node("agent", call_model)
agent_graph_builder.add_node("tools", call_tools)
agent_graph_builder.add_edge(START, "agent")
agent_graph_builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
agent_graph_builder.add_edge("tools", "agent")
agent_graph = agent_graph_builder.compile()

print(agent_graph.get_graph().draw_mermaid())

In [ ]:
def run_agent_graph(question: str) -> str:
    init = {"messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]}
    final_state = agent_graph.invoke(init, config={"recursion_limit": MAX_TOOL_ROUNDS * 2 + 2})
    return final_state["messages"][-1]["content"] or ""


print(run_agent_graph("Combien de clients ont un encours de credit superieur a 50000 ?"))

## 6. Approche C — pipeline guidé (retrouve → génère → valide → exécute)

Le *tool calling* libre (A, B) est flexible mais imprévisible : rien n'empêche le
modèle de sauter la vérification, d'inventer une colonne, ou de tourner en rond.
Un pipeline guidé fixe l'**ordre** dans le code et ne demande au modèle que de
remplir une case précise (le SQL) — même philosophie que `docmaker/eval/runtime.py`
du dépôt : jamais de réponse plausible mais fausse, une **abstention motivée** après
un nombre borné d'échecs de validation.

Graphe : `retrieve` (recherche sémantique, sans LLM) → `generate_sql` (1 appel LLM,
JSON contraint) → `validate_and_execute` (SELECT seul + tables connues + exécution)
→ si erreur et tentatives restantes, retour à `generate_sql` avec le message
d'erreur ; sinon `finalize` (réponse en langage naturel) ou `abstain`.

In [ ]:
class PipelineState(TypedDict):
    question: str
    attempts: int
    context: str
    sql: str
    error: str
    rows: list[dict]
    answer: str


def retrieve(state: PipelineState) -> dict:
    hits = search_catalog(state["question"], top_k=3, entity_type="table")
    lines = []
    for hit in hits:
        meta = CATALOG[hit["fqn"]]
        cols = ", ".join(f"{c['name']} ({c['type']})" for c in meta["columns"])
        lines.append(f"- table sqlite `{meta['sqlite_table']}` [{hit['fqn']}] : {meta['comment']} | colonnes: {cols}")
    return {"context": "\n".join(lines), "attempts": 0, "error": ""}

In [ ]:
_JSON_FENCE = "```"


def _extract_json_object(raw: str) -> dict:
    """Meme tolerance que `docmaker/llm.py::_strip` : on ne suppose pas que le modele
    respecte un mode JSON strict (tous les backends OpenAI-compatibles ne le
    supportent pas de la meme facon), on nettoie un eventuel bloc de code et on
    isole le premier objet.
    """
    s = raw.strip()
    if s.startswith(_JSON_FENCE):
        s = s.strip("`").removeprefix("json").strip()
    i, j = s.find("{"), s.rfind("}")
    return json.loads(s[i : j + 1] if 0 <= i < j else s)


def generate_sql(state: PipelineState) -> dict:
    prompt = f"Tables disponibles :\n{state['context']}\n\nQuestion : {state['question']}\n"
    if state.get("error"):
        prompt += (
            f"\nLa proposition precedente a echoue :\n{state.get('sql', '')}\n"
            f"Erreur : {state['error']}\nCorrige le SQL en consequence."
        )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "Tu ecris une seule requete SQL SQLite, en SELECT uniquement, sur les "
                "tables sqlite listees (utilise leur nom sqlite, pas le FQN entre crochets). "
                'Reponds avec UN SEUL objet JSON, sans texte autour : {"sql": "..."}'
            )},
            {"role": "user", "content": prompt},
        ],
    )
    raw = response.choices[0].message.content or ""
    try:
        sql = _extract_json_object(raw)["sql"]
    except Exception as e:
        return {"sql": "", "error": f"reponse non JSON valide : {e}", "attempts": state["attempts"] + 1}
    return {"sql": sql, "attempts": state["attempts"] + 1}

In [ ]:
def validate_and_execute(state: PipelineState) -> dict:
    sql = state["sql"]
    if not sql:
        return {"error": state.get("error") or "sql vide"}
    try:
        _ensure_select_only(sql, "sqlite")
    except ReadOnlyViolation as e:
        return {"error": str(e)}

    referenced = {t.name.lower() for t in sqlglot.parse_one(sql, dialect="sqlite").find_all(exp.Table)}
    unknown = referenced - set(_SQLITE_TO_FQN)
    if unknown:
        return {"error": f"table(s) inconnue(s) du catalogue : {sorted(unknown)}"}

    outcome = run_sql(sql)
    if "error" in outcome:
        return {"error": outcome["error"]}
    return {"rows": outcome["rows"], "error": ""}


def route_after_validation(state: PipelineState) -> str:
    if not state.get("error"):
        return "finalize"
    if state["attempts"] >= MAX_SQL_ATTEMPTS:
        return "abstain"
    return "generate_sql"


def finalize(state: PipelineState) -> dict:
    response = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "Resume ces resultats SQL en une reponse courte, en francais."},
        {"role": "user", "content": f"Question : {state['question']}\nResultats : {json.dumps(state['rows'], default=str)}"},
    ])
    return {"answer": response.choices[0].message.content or ""}


def abstain(state: PipelineState) -> dict:
    return {"answer": f"abstention apres {state['attempts']} tentative(s) : {state['error']}"}


guided_builder = StateGraph(PipelineState)
guided_builder.add_node("retrieve", retrieve)
guided_builder.add_node("generate_sql", generate_sql)
guided_builder.add_node("validate_and_execute", validate_and_execute)
guided_builder.add_node("finalize", finalize)
guided_builder.add_node("abstain", abstain)
guided_builder.add_edge(START, "retrieve")
guided_builder.add_edge("retrieve", "generate_sql")
guided_builder.add_edge("generate_sql", "validate_and_execute")
guided_builder.add_conditional_edges(
    "validate_and_execute", route_after_validation,
    {"finalize": "finalize", "generate_sql": "generate_sql", "abstain": "abstain"},
)
guided_builder.add_edge("finalize", END)
guided_builder.add_edge("abstain", END)
guided_pipeline = guided_builder.compile()

print(guided_pipeline.get_graph().draw_mermaid())

In [ ]:
def run_guided(question: str, verbose: bool = True) -> str:
    state: PipelineState = {
        "question": question, "attempts": 0, "context": "", "sql": "",
        "error": "", "rows": [], "answer": "",
    }
    answer = ""
    for event in guided_pipeline.stream(state, config={"recursion_limit": 25}):
        for node, update in event.items():
            if verbose:
                print(f"[{node}] {update}")
            if "answer" in update:
                answer = update["answer"]
    return answer


print(run_guided("Quel est le solde du compte 103 ?"))

## 7. Comparer les trois approches

| Critère                          | A — boucle manuelle          | B — LangGraph (même agent)     | C — pipeline guidé (LangGraph)                |
| ---------------------------------- | ------------------------------ | --------------------------------- | ------------------------------------------------ |
| Dépendance en plus                 | aucune                          | `langgraph`                       | `langgraph`                                       |
| Liberté du modèle                  | totale                          | totale (identique à A)            | bridée : slots précis (le SQL, la synthèse finale) |
| Prévisibilité de l'ordre d'appel   | décidée par le modèle           | décidée par le modèle             | décidée par le code                               |
| Abstention motivée si échec        | non — peut halluciner un résultat | non — idem A                    | oui, après `MAX_SQL_ATTEMPTS`                     |
| Où LangGraph apporte quelque chose | —                               | rien de plus qu'A ici             | branchement conditionnel, retry borné, état typé, `stream()` pour observer chaque étape |
| Coût                                | 1 appel LLM par tour d'outil    | idem A                            | appels bornés : retrieve gratuit, ≤`MAX_SQL_ATTEMPTS` génération, 1 synthèse |

**À retenir** : LangGraph n'ajoute rien en A vs B — c'est un `while` avec des noms en
plus. Sa valeur apparaît en C, quand le contrôle de flux (qui appeler, dans quel
ordre, combien de fois réessayer) doit être garanti par le code plutôt que par le bon
vouloir du modèle. Le choix entre "agent libre" et "pipeline guidé" est un choix de
produit (couverture large et imprévisible vs couverture étroite et fiable), pas un
choix technique — LangGraph sert les deux.

## 8. Quoi tweaker

| Variable/fonction        | Effet                                                                 |
| ------------------------- | ---------------------------------------------------------------------- |
| `PROVIDER`                | `"ollama"` (local) ↔ `"openrouter"` (cloud) — même code, seul `base_url` change |
| `OLLAMA_MODEL` / `OPENROUTER_MODEL` | le modèle doit supporter le tool calling pour A/B ; C n'en a pas besoin (JSON best-effort, comme `docmaker/llm.py`) |
| `DB_BACKEND`              | `"demo_sqlite"` ↔ `"oracle"` — brancher un vrai datamart en lecture seule |
| `MAX_RESULT_ROWS` / `QUERY_TIMEOUT_SECONDS` | les deux garde-fous non négociables de `run_sql`            |
| `MAX_TOOL_ROUNDS` / `MAX_SQL_ATTEMPTS` | bornes anti-boucle des approches A/B et C respectivement       |
| `CATALOG` / `LINEAGE_COLUMNS` / `JOIN_EDGES` | remplacer par un vrai `build/catalog.json`/`build/lineage.json`/`build/joins.json` du pipeline |
| `TOOLS` / `TOOL_FUNCS`    | ajouter un outil (ex. `get_priority` sur `build/priority.json`) : une entrée dans chaque dict, rien d'autre à changer |
| `SYSTEM_PROMPT`           | resserrer/assouplir la liberté du modèle dans les approches A/B         |